# 🤖 OnePilot — Sprint 9 Benchmark
## Multi-Agent RAG : Orchestrateur + MultiQuery + DirectSQL

**Objectifs :**
- Valider le routage Orchestrateur (DirectSQL / MultiQuery / ReAct+)
- Mesurer la latence des questions composées (ET / VS)
- Vérifier la couverture SQL des entités clés
- Comparer les métriques Sprint 8.6 → Sprint 9


In [ ]:
import requests
import time
import re
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML

BASE_URL  = 'http://onepilot_api:8000'
SOURCE_ID = '85a0ef4b-d9af-494f-b24f-ff710c21ba43'

print("✅ Imports OK")
print(f"   BASE_URL  : {BASE_URL}")
print(f"   SOURCE_ID : {SOURCE_ID}")


In [ ]:
# ── Fonctions de validation ──────────────────────────────────────────────────

def extract_key_entities(question: str) -> dict:
    """Extrait les entités clés d'une question pour valider la couverture SQL."""
    entities = {}
    amounts = re.findall(r'\b\d{3,}\b', question)
    if amounts: entities['amounts'] = amounts
    years = re.findall(r'\b(?:19|20)\d{2}\b', question)
    if years: entities['years'] = years
    currencies = re.findall(r'\b(TND|EUR|USD|GBP|CHF)\b', question, re.IGNORECASE)
    if currencies: entities['currencies'] = [c.upper() for c in currencies]
    known_banks = ['bnp', 'banque postale', 'société générale', 'stb', 'biat', 'attijari']
    for bank in known_banks:
        if bank in question.lower():
            entities.setdefault('banks', []).append(bank)
    return entities

def _normalize_str(s):
    """Normalise une chaîne : minuscules + supprime accents."""
    import unicodedata
    return ''.join(c for c in unicodedata.normalize('NFD', s.lower())
                   if unicodedata.category(c) != 'Mn')

def validate_sql_coverage(question: str, sql: str) -> tuple:
    """Vérifie que le SQL couvre les entités clés — insensible accents/casse."""
    if not sql: return False, 0.0, ['SQL vide']
    entities = extract_key_entities(question)
    if not entities: return True, 1.0, []
    sql_norm = _normalize_str(sql)
    missing, total, covered = [], 0, 0
    for etype, values in entities.items():
        for val in values:
            total += 1
            if _normalize_str(val) in sql_norm: covered += 1
            else: missing.append(f'{etype}:{val}')
    score = covered / total if total > 0 else 1.0
    return score >= 0.6, round(score, 2), missing

def query_sprint9(question, verbose=True):
    """Appelle l'API et mesure méthode, agent, temps, couverture SQL."""
    t0 = time.time()
    try:
        r = requests.post(
            f'{BASE_URL}/orchestrator/query',
            json={'question': question, 'source_id': SOURCE_ID,
                  'dialect': 'mssql', 'verbose': False},
            timeout=180
        )
        elapsed = round((time.time() - t0) * 1000)
        data    = r.json()
        sql     = data.get('sql', '')
        method  = data.get('method', 'unknown')
        agent   = data.get('agent_type', 'unknown')
        iters   = data.get('iterations', 1)
        sqls    = data.get('sqls', [sql] if sql else [])
        subs    = data.get('sub_queries', [])

        sql_valid, sql_score, missing = validate_sql_coverage(question, sql)
        success = bool(sql and 'ERROR' not in sql.upper() and sql_valid)

        if verbose:
            is_multi = agent == 'multi_query' or len(sqls) > 1
            if is_multi:
                icon = '🔀'
                n_ok = sum(1 for s in subs if s.get('success', False)) if subs else len([s for s in sqls if s])
                n_total = len(subs) if subs else len(sqls)
                tag = f'MULTI({n_ok}/{n_total})'
            elif 'direct' in method:
                icon, tag = '⚡', 'DIRECT'
            else:
                icon, tag = '🔄', 'REACT '
            ok = '✅' if success else '⚠️ '
            cov = f'cov={sql_score:.0%}' if missing else 'cov=✅'
            print(f'  {ok} {icon} [{tag:10}] {elapsed:>6}ms  {cov}  {question[:55]}')
            if missing: print(f'         ⚠️ manquants: {missing}')

        return {
            'question': question, 'sql': sql, 'sqls': sqls,
            'method': method, 'agent': agent,
            'iterations': iters, 'elapsed_ms': elapsed,
            'success': success, 'sql_score': sql_score,
            'missing': missing, 'is_multi': len(sqls) > 1,
            'n_sub_ok': sum(1 for s in subs if s.get('success')) if subs else (1 if success else 0),
            'n_sub_total': len(subs) if subs else 1,
        }
    except Exception as e:
        if verbose: print(f'  ❌ ERREUR : {e}')
        return {'question': question, 'sql': '', 'sqls': [], 'method': 'error',
                'agent': 'error', 'iterations': 0, 'elapsed_ms': 0,
                'success': False, 'sql_score': 0.0, 'missing': [str(e)],
                'is_multi': False, 'n_sub_ok': 0, 'n_sub_total': 1}

# Test connexion
try:
    r = requests.get(f'{BASE_URL}/health', timeout=5)
    print(f'✅ API connectée — status={r.status_code}')
except Exception as e:
    print(f'❌ Connexion échouée : {e}')


In [ ]:
# ── TIER 1 : Questions DirectSQL (bypass LLM) ────────────────────────────────
print("=" * 65)
print("  TIER 1 — DirectSQL (objectif < 500ms)")
print("=" * 65)

TIER1_QUESTIONS = [
    "liste les devises",
    "financements actifs",
    "solde bancaire",
    "liste les banques",
    "liste les sociétés",
    "taux de change",
    "comptes avec leur solde",
    "utilisateurs bloqués",
    "transactions TND supérieures à 10000 La Banque Postale 2024",
    "transactions EUR supérieures à 50000 pour les comptes BNP en 2023",
    "financements dont la maturité dépasse 5 ans par banque et groupe de sociétés",
    "utilisateurs avec accès à plus d'une société",
    "total des transactions par devise en 2024",
    "financements BNP",
    "financements Société Générale",
    "comptes BNP",
    "comptes Société Générale",
    "utilisateurs bloqués et leurs sociétés associées",
    "flux de trésorerie du mois",
    "financements arrivant à échéance",
]

results_tier1 = []
for q in TIER1_QUESTIONS:
    r = query_sprint9(q)
    results_tier1.append(r)

# Résumé Tier 1
ok1    = [r for r in results_tier1 if r['success']]
direct = [r for r in results_tier1 if 'direct' in r['method']]
avg_t1 = round(sum(r['elapsed_ms'] for r in results_tier1) / len(results_tier1))
print(f"\n  Résultats : {len(ok1)}/{len(results_tier1)} succès")
print(f"  DirectSQL : {len(direct)}/{len(results_tier1)}")
print(f"  Temps moyen : {avg_t1}ms (objectif < 500ms)")


In [ ]:
# ── TIER 2 : Questions composées ET (MultiQuery fusion) ──────────────────────
print("=" * 65)
print("  TIER 2 — MultiQuery ET (objectif < 3s)")
print("=" * 65)

TIER2_ET_QUESTIONS = [
    "total transactions BNP 2024 et solde bancaire par société",
    "liste les financements actifs et les comptes associés",
    "liste les banques et les devises associées",
    "solde bancaire par société et total des financements actifs",
    "liste les utilisateurs actifs et leurs droits d'accès",
    "solde des comptes EUR et total des transactions USD",
    "flux de trésorerie du mois et financements arrivant à échéance",
    "utilisateurs bloqués et leurs sociétés associées",
]

results_tier2_et = []
for q in TIER2_ET_QUESTIONS:
    r = query_sprint9(q)
    results_tier2_et.append(r)

ok2et  = [r for r in results_tier2_et if r['success']]
multi2 = [r for r in results_tier2_et if r['is_multi']]
avg_t2 = round(sum(r['elapsed_ms'] for r in results_tier2_et) / len(results_tier2_et))
print(f"\n  Résultats : {len(ok2et)}/{len(results_tier2_et)} succès")
print(f"  MultiQuery : {len(multi2)}/{len(results_tier2_et)}")
print(f"  Temps moyen : {avg_t2}ms (objectif < 3000ms)")


In [ ]:
# ── TIER 3 : Questions composées VS/compare (MultiQuery comparaison) ──────────
print("=" * 65)
print("  TIER 3 — MultiQuery VS/compare (objectif < 3s)")
print("=" * 65)

TIER3_VS_QUESTIONS = [
    "compare les devises disponibles vs les taux de change",
    "compare les financements ouverts vs les financements clôturés",
    "compare les comptes BNP vs les comptes Société Générale",
    "compare les financements BNP vs les financements Société Générale",
    "compare les utilisateurs actifs vs les utilisateurs bloqués",
]

results_tier3_vs = []
for q in TIER3_VS_QUESTIONS:
    r = query_sprint9(q)
    results_tier3_vs.append(r)

ok3vs  = [r for r in results_tier3_vs if r['success']]
multi3 = [r for r in results_tier3_vs if r['is_multi']]
avg_t3 = round(sum(r['elapsed_ms'] for r in results_tier3_vs) / len(results_tier3_vs))
print(f"\n  Résultats : {len(ok3vs)}/{len(results_tier3_vs)} succès")
print(f"  MultiQuery : {len(multi3)}/{len(results_tier3_vs)}")
print(f"  Temps moyen : {avg_t3}ms (objectif < 3000ms)")


# ── RÉSUMÉ FINAL Sprint 9 ────────────────────────────────────────────────────
print("=" * 65)
print("  RÉSUMÉ FINAL — Sprint 9 Multi-Agent RAG")
print("=" * 65)

all_results = results_tier1 + results_tier2_et + results_tier3_vs
all_ok      = [r for r in all_results if r['success']]
all_direct  = [r for r in all_results if 'direct' in r['method']]
all_multi   = [r for r in all_results if r['is_multi']]
all_react   = [r for r in all_results if 'react' in r['method'] and not r['is_multi']]
avg_total   = round(sum(r['elapsed_ms'] for r in all_results) / len(all_results))

print(f"\n  Questions testées  : {len(all_results)}")
print(f"  Succès total       : {len(all_ok)}/{len(all_results)} ({len(all_ok)/len(all_results):.0%})")
print(f"\n  Par agent :")
print(f"    ⚡ DirectSQL      : {len(all_direct)} questions")
print(f"    🔀 MultiQuery     : {len(all_multi)} questions")
print(f"    🔄 ReAct+         : {len(all_react)} questions")
print(f"\n  Latence moyenne   : {avg_total}ms")
print(f"  Tier 1 (direct)   : {round(sum(r['elapsed_ms'] for r in results_tier1)/len(results_tier1))}ms")
print(f"  Tier 2 (ET)       : {round(sum(r['elapsed_ms'] for r in results_tier2_et)/len(results_tier2_et))}ms")
print(f"  Tier 3 (VS)       : {round(sum(r['elapsed_ms'] for r in results_tier3_vs)/len(results_tier3_vs))}ms")

print(f"\n  Amélioration vs Sprint 8.6 :")
print(f"    Questions composées : ❌ non supporté → ✅ {len(all_multi)} questions MultiQuery")
print(f"    Latence P95         : ~20s ReAct → ~{round(sum(r['elapsed_ms'] for r in results_tier2_et+results_tier3_vs)/len(results_tier2_et+results_tier3_vs)/1000, 1)}s MultiQuery")
print(f"    Hallucination LLM   : réduite via bypass Direct SQL étendu")


In [ ]:
# ── GRAPHIQUES Sprint 9 ──────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Recalcul des agrégats
all_results = results_tier1 + results_tier2_et + results_tier3_vs
all_direct  = [r for r in all_results if 'direct' in r['method']]
all_multi   = [r for r in all_results if r['is_multi']]
all_react   = [r for r in all_results if 'react' in r['method'] and not r['is_multi']]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('OnePilot Sprint 9 — Multi-Agent RAG Benchmark', fontsize=14, fontweight='bold')

# ── Graphe 1 : Latence moyenne par tier ──────────────────────────────────────
tiers_lbl = ['Tier 1\nDirectSQL', 'Tier 2\nMultiQuery ET', 'Tier 3\nMultiQuery VS']
def safe_avg(lst): return round(sum(r['elapsed_ms'] for r in lst) / len(lst)) if lst else 0
latences = [safe_avg(results_tier1), safe_avg(results_tier2_et), safe_avg(results_tier3_vs)]
colors_t = ['#2ecc71', '#3498db', '#9b59b6']
bars = axes[0].bar(tiers_lbl, latences, color=colors_t, alpha=0.85, width=0.5)
axes[0].axhline(y=500,  color='green',  linestyle='--', alpha=0.6, label='Objectif T1 (500ms)')
axes[0].axhline(y=3000, color='orange', linestyle='--', alpha=0.6, label='Objectif T2/T3 (3s)')
axes[0].set_title('Latence moyenne par tier')
axes[0].set_ylabel('ms')
axes[0].legend(fontsize=8)
for bar, val in zip(bars, latences):
    if val > 0:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                     f'{val}ms', ha='center', va='bottom', fontsize=10, fontweight='bold')

# ── Graphe 2 : Distribution agents (bar, pas pie) ────────────────────────────
agent_names  = ['DirectSQL', 'MultiQuery', 'ReAct+']
agent_values = [len(all_direct), len(all_multi), len(all_react)]
agent_colors = ['#2ecc71', '#3498db', '#e74c3c']
bars2 = axes[1].bar(agent_names, agent_values, color=agent_colors, alpha=0.85, width=0.5)
axes[1].set_title('Distribution par agent')
axes[1].set_ylabel('Nombre de questions')
for bar, val in zip(bars2, agent_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')

# ── Graphe 3 : Taux de succès par tier ───────────────────────────────────────
tiers_ok    = [len([r for r in results_tier1 if r['success']]),
               len([r for r in results_tier2_et if r['success']]),
               len([r for r in results_tier3_vs if r['success']])]
tiers_total = [len(results_tier1), len(results_tier2_et), len(results_tier3_vs)]
tiers_fail  = [t - ok for t, ok in zip(tiers_total, tiers_ok)]
x = np.arange(len(tiers_lbl))
axes[2].bar(x, tiers_ok,  label='Succès', color='#2ecc71', alpha=0.85)
axes[2].bar(x, tiers_fail, bottom=tiers_ok, label='Échec', color='#e74c3c', alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels(['Tier 1\nDirectSQL', 'Tier 2\nET', 'Tier 3\nVS'])
axes[2].set_title('Succès par tier')
axes[2].set_ylabel('Nombre de questions')
axes[2].legend()
for i, (ok, total) in enumerate(zip(tiers_ok, tiers_total)):
    axes[2].text(i, total + 0.1, f'{ok}/{total}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('benchmark_sprint9.png', dpi=150, bbox_inches='tight')
plt.close()
print("Graphique sauvegarde : benchmark_sprint9.png")
print(f"Tier 1: {tiers_ok[0]}/{tiers_total[0]} succes | {latences[0]}ms moy")
print(f"Tier 2: {tiers_ok[1]}/{tiers_total[1]} succes | {latences[1]}ms moy")
print(f"Tier 3: {tiers_ok[2]}/{tiers_total[2]} succes | {latences[2]}ms moy")
